# Экспорт эталонных формулировок из Colab ревьюера → JSONL для движка

Этот ноутбук предназначен для **Google Colab** (или локального Jupyter): загрузите **эталонный `.ipynb` старшего ревьюера** и сгенерируйте файл `reviewer_reference_examples.jsonl`.

**Важно:** движок использует эти строки только как **стилистические опоры** при улучшении комментариев (RAG-подсказки). Они **не** являются рубрикой, **не** ground truth и **не** требуют от студента копировать эталонное решение — в промпте LLM это зафиксировано явно.

Формат строки JSONL (как у основного корпуса примеров):
- `example_id` — произвольный id
- `criterion_code` — код критерия из вашей criteria map (как в `configs/criteria_maps/*.json`)
- `text` — короткий эталонный комментарий или фрагмент обоснования **на русском**
- `tags` — опционально, например `["colab", "senior_reference"]`

В `.env` сервиса: `ENABLE_RETRIEVAL=true`, `ENABLE_REVIEWER_REFERENCE_EXAMPLES=true`, `REVIEWER_REFERENCE_EXAMPLES_PATH=...`

In [ ]:
# Colab: раскомментируйте при необходимости
# !pip -q install nbformat

import json
from pathlib import Path

import nbformat

# Загрузите эталонный ipynb в Colab через файловую панель, затем укажите путь:
REFERENCE_IPYNB = Path("senior_reference.ipynb")  # измените имя
OUTPUT_JSONL = Path("reviewer_reference_examples.jsonl")

# Сопоставление: фрагмент заголовка markdown-ячейки (нижний регистр) -> criterion_code
SECTION_HINT_TO_CRITERION = {
    "вывод": "NB_CONCLUSION",
    "визуал": "NB_VIZ",
    # добавьте свои подстроки и коды критериев под ваш practicum map
}

DEFAULT_CRITERION = "NB_GENERAL"


def infer_criterion(source: str) -> str:
    low = source.lower()
    for hint, code in SECTION_HINT_TO_CRITERION.items():
        if hint in low:
            return code
    return DEFAULT_CRITERION


def cells_to_jsonl(nb_path: Path, out_path: Path) -> int:
    nb = nbformat.read(nb_path, as_version=4)
    rows = []
    for i, cell in enumerate(nb.cells):
        src = "".join(cell.get("source", ""))
        if cell.cell_type == "markdown" and len(src.strip()) > 40:
            rows.append(
                {
                    "example_id": f"ref_md_{i}",
                    "criterion_code": infer_criterion(src),
                    "text": src.strip()[:1200],
                    "tags": ["colab", "senior_reference", "markdown_cell"],
                }
            )
        if cell.cell_type == "code" and cell.get("outputs"):
            for out in cell.outputs:
                data = getattr(out, "data", None) or {}
                t = data.get("text/plain")
                if isinstance(t, list):
                    t = "".join(t)
                if isinstance(t, str) and len(t.strip()) > 20:
                    rows.append(
                        {
                            "example_id": f"ref_out_{i}",
                            "criterion_code": infer_criterion(src),
                            "text": t.strip()[:800],
                            "tags": ["colab", "senior_reference", "output"],
                        }
                    )
    out_path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows), encoding="utf-8")
    return len(rows)


if not REFERENCE_IPYNB.is_file():
    raise FileNotFoundError(f"Нет файла {REFERENCE_IPYNB} — загрузите эталонный ipynb")

n = cells_to_jsonl(REFERENCE_IPYNB, OUTPUT_JSONL)
print(f"Записано строк: {n} -> {OUTPUT_JSONL.resolve()}")
print("Скачайте JSONL и положите на сервер с review assistant; задайте REVIEWER_REFERENCE_EXAMPLES_PATH")

## Ручное дополнение

Автоматический разбор ячеек — заготовка. Надёжнее вручную добавить в JSONL 1–3 эталонные формулировки **на критерий** (короткий текст, как пишет старший ревьюер при `fail`/`warn`). Отредактируйте файл или добавьте ниже ячейку с `json.dumps` своих объектов.